# Stage 2 - ViT5-base tự train Stage 1, fine-tune domain Y tế

Load checkpoint Stage 1 do mình tự fine-tune từ `VietAI/vit5-base` trên VietNews, rồi fine-tune tiếp trên dữ liệu y tế.

Flow Stage 2:
- Cell 1-2: setup, import, config.
- Cell 3-5: mount Drive, load dataset y tế, augmentation an toàn.
- Cell 6: fine-tune Stage 2 với early stopping và checkpoint tốt nhất.
- Cell 7: đánh giá SFT bằng ROUGE, BERTScore, number recall, hallucination, length compliance.
- Cell 8: sinh nhiều ứng viên và rerank theo số liệu + độ dài.
- Cell 9: đánh giá cuối và export JSON/CSV để so sánh 3 model.

Notebook này độc lập với `stage2_vit5_checkpoint.ipynb`.


## Cell 1 - Setup

In [1]:
import sys
import subprocess
import importlib.metadata as md

PACKAGES = [
    "transformers==4.46.3",
    "datasets==3.1.0",
    "evaluate==0.4.3",
    "accelerate==1.1.1",
    "rouge-score==0.1.2",
    "bert-score==0.3.13",
    "sentencepiece==0.2.0",
]

missing = []
for spec in PACKAGES:
    name = spec.split("==")[0]
    dist_name = "rouge-score" if name == "rouge-score" else name
    try:
        md.version(dist_name)
    except md.PackageNotFoundError:
        missing.append(spec)

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *missing])
else:
    print("Đã có đủ package cần thiết.")

## Cell 2 - Imports & Config

In [ ]:
MODEL_LABEL = 'ViT5-base self Stage1 -> Stage2'
STAGE1_CKPT = '/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_1/models/stage1_vit5/final_model'
BASE_DIR = '/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_2'
RUN_NAME = 'vit5_base_stage2'

import os
import re
import gc
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

CONFIG = {
    "MODEL_LABEL": MODEL_LABEL,
    "STAGE1_CKPT": STAGE1_CKPT,
    "BASE_DIR": BASE_DIR,
    "RUN_NAME": RUN_NAME,

    "USE_EXISTING_AUGMENTED": False,
    "AUGMENT_PER_SAMPLE": 2,
    "USE_BACK_TRANSLATE": False,
    "SYNONYM_REPLACE_PROB": 0.18,

    "MAX_INPUT_LEN": 1024,
    "MAX_TARGET_LEN": 512,
    "GEN_MAX_LEN": 320,
    "GEN_MIN_LEN": 64,

    "EPOCHS": 6,
    "LR": 3e-5,
    "TRAIN_BATCH": 4,
    "EVAL_BATCH": 8,
    "GRAD_ACCUM": 4,
    "WEIGHT_DECAY": 0.01,
    "WARMUP_RATIO": 0.06,
    "LABEL_SMOOTH": 0.1,
    "PATIENCE": 2,

    "NUM_BEAMS": 4,
    "NO_REPEAT_NGRAM": 3,
    "LENGTH_PENALTY": 1.0,
    "NUM_CANDIDATES": 5,
    "DIVERSITY_PENALTY": 0.5,

    "MIN_RATIO": 0.20,
    "MAX_RATIO": 0.25,
    "TARGET_RATIO": 0.225,
    "W_NUMBER": 0.50,
    "W_LENGTH": 0.30,
    "W_HALLUCINATION": 0.20,
}

USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
USE_FP16 = bool(torch.cuda.is_available() and not USE_BF16)

BASE_DIR = Path(CONFIG["BASE_DIR"])
OUT_DIR = BASE_DIR / "runs" / CONFIG["RUN_NAME"]
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
BEST_DIR = OUT_DIR / "best_model"
RESULTS_DIR = OUT_DIR / "results"
for d in (OUT_DIR, CHECKPOINT_DIR, BEST_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Model:", CONFIG["MODEL_LABEL"])
print("Stage 1 ckpt:", CONFIG["STAGE1_CKPT"])
print("Output:", OUT_DIR)
print("Precision:", "bf16" if USE_BF16 else ("fp16" if USE_FP16 else "fp32"))
CONFIG

## Cell 3 - Mount Drive & Find Dataset

In [ ]:
from google.colab import drive

# Mount without deleting or unmounting a possibly live Drive tree.
drive.mount('/content/drive')

CANDIDATES = [
    Path('/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/data'),
    Path('/content/drive/MyDrive/NLP_CK/data'),
    Path('/content/drive/MyDrive/data'),
    Path.cwd() / 'data',
    Path.cwd().parent / 'data',
]

DATA_DIR = None
for candidate in CANDIDATES:
    has_splits = (
        (candidate / 'train' / 'dataset.jsonl').exists()
        and (candidate / 'val' / 'dataset.jsonl').exists()
        and (candidate / 'test' / 'dataset.jsonl').exists()
    )
    if has_splits:
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    print("Không tự tìm thấy DATA_DIR. Các path đã thử:")
    for candidate in CANDIDATES:
        print(" -", candidate)
    raise FileNotFoundError("Sửa DATA_DIR thủ công để trỏ tới folder data chứa train/val/test.")

stage1_path = Path(CONFIG['STAGE1_CKPT'])
if not stage1_path.exists():
    raise FileNotFoundError(
        f"Chưa thấy Stage 1 checkpoint ở {stage1_path}. "
        "Hãy chạy xong Stage 1 và kiểm tra CONFIG['STAGE1_CKPT']."
    )

print("DATA_DIR:", DATA_DIR)
print("Có train_augmented:", (DATA_DIR / 'train' / 'train_augmented.jsonl').exists())

## Cell 4 - Load Dataset Y Tế

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def normalize_record(row):
    if not isinstance(row, dict):
        raise ValueError("Each dataset row must be a JSON object")
    for field in ("source", "summary"):
        if not isinstance(row.get(field), str) or not row[field].strip():
            raise ValueError(f"Row {row.get('id', '<unknown>')!r}: {field} must be nonempty text")
    source, summary = row["source"].strip(), row["summary"].strip()
    return {
        "id": str(row.get("id") or ""), "url": str(row.get("url") or ""),
        "category": row.get("category", ""), "quality_tag": row.get("quality_tag", ""),
        "source": source, "summary": summary,
        "src_word_count": len(source.split()), "tgt_word_count": len(summary.split()),
        "length_ratio": len(summary.split()) / len(source.split()),
    }


def validate_splits(splits):
    """Reject exact overlap by normalized source, nonempty ID or URL; do not alter splits."""
    import unicodedata
    seen = {}
    for split, rows in splits.items():
        if not rows:
            raise ValueError(f"Empty split: {split}")
        for row in rows:
            key_text = " ".join(unicodedata.normalize("NFC", row["source"]).casefold().split())
            keys = [("source", key_text)]
            keys += [(field, str(row.get(field) or "").strip()) for field in ("id", "url")
                     if str(row.get(field) or "").strip()]
            for key in keys:
                previous = seen.get(key)
                if previous is not None and previous != split:
                    raise ValueError(f"Dataset overlap between {previous} and {split} by {key[0]}")
                seen[key] = split



train_raw = [normalize_record(r) for r in load_jsonl(DATA_DIR / 'train' / 'dataset.jsonl')]
val_raw = [normalize_record(r) for r in load_jsonl(DATA_DIR / 'val' / 'dataset.jsonl')]
test_raw = [normalize_record(r) for r in load_jsonl(DATA_DIR / 'test' / 'dataset.jsonl')]

validate_splits({"train": train_raw, "val": val_raw, "test": test_raw})

print(f"Raw train/val/test: {len(train_raw):,} / {len(val_raw):,} / {len(test_raw):,}")
print("Category train:", dict(Counter(r['category'] for r in train_raw)))
print("Avg ratio train:", round(float(np.mean([r['length_ratio'] for r in train_raw])), 4))

## Cell 5 - Augmentation An Toàn

In [ ]:
# Orthographic variant only; unchanged variants are not added as training duplicates.
MEDICAL_SYNONYMS = {'vaccine': ['vắc xin']}

NUM_RE = re.compile(r'\d+(?:[.,]\d+)*')

def extract_numbers(text):
    return set(NUM_RE.findall(text or ''))

def synonym_replace_medical(text, prob=0.18, seed=None):
    # Thay từ đồng nghĩa nhẹ, không động vào chữ số.
    rng = random.Random(seed)
    output = text
    for src, replacements in MEDICAL_SYNONYMS.items():
        if src in output and rng.random() < prob:
            repl = rng.choice(replacements)
            output = re.sub(rf'(?<!\w){re.escape(src)}(?!\w)', repl, output, flags=re.IGNORECASE)
    return output

def back_translate(text):
    # Placeholder an toàn. Mặc định tắt để tránh phụ thuộc model dịch nặng
    # và tránh làm sai số liệu y tế. Nếu muốn back-translation thật,
    # bật CONFIG['USE_BACK_TRANSLATE']=True và thay thân hàm bằng pipeline vi->en->vi.
    if CONFIG['USE_BACK_TRANSLATE']:
        raise NotImplementedError('Back translation is not implemented; keep USE_BACK_TRANSLATE=False')
    return text

def augment_dataset(records, n_aug=2):
    augmented = []
    for i, row in enumerate(records):
        augmented.append(row)
        src_nums = extract_numbers(row['source'])
        seen_sources = {' '.join(row['source'].split())}
        for j in range(n_aug):
            new_source = synonym_replace_medical(
                row['source'],
                CONFIG['SYNONYM_REPLACE_PROB'],
                seed=SEED + i * 17 + j,
            )
            new_source = back_translate(new_source)
            if extract_numbers(new_source) != src_nums:
                continue
            source_key = ' '.join(new_source.split())
            if source_key in seen_sources:
                continue
            seen_sources.add(source_key)
            aug = dict(row)
            aug['id'] = f"{row['id']}_safe_aug_{j+1}"
            aug['source'] = new_source
            aug['src_word_count'] = len(new_source.split())
            aug['length_ratio'] = len(aug['summary'].split()) / max(len(new_source.split()), 1)
            aug['quality_tag'] = row.get('quality_tag', 'augmented') or 'augmented'
            augmented.append(aug)
    return augmented

aug_path = DATA_DIR / 'train' / 'train_augmented.jsonl'
if CONFIG['USE_EXISTING_AUGMENTED'] and aug_path.exists():
    train_stage2 = [normalize_record(r) for r in load_jsonl(aug_path)]
    print(f"Dùng file augment sẵn: {aug_path}")
else:
    train_stage2 = augment_dataset(train_raw, n_aug=CONFIG['AUGMENT_PER_SAMPLE'])
    print("Tự sinh augment trong notebook.")

print(f"Stage 2 train: {len(train_stage2):,} | val: {len(val_raw):,} | test: {len(test_raw):,}")
print("Avg train source words:", round(float(np.mean([r['src_word_count'] for r in train_stage2])), 1))
print("Avg train target words:", round(float(np.mean([r['tgt_word_count'] for r in train_stage2])), 1))
print("Avg train ratio:", round(float(np.mean([r['length_ratio'] for r in train_stage2])), 4))
validate_splits({"train": train_stage2, "val": val_raw, "test": test_raw})


## Cell 6 - Tokenizer, Model & Stage 2 Fine-tune

In [7]:
import sys
import huggingface_hub.hf_api

# 1. Tự định nghĩa lại Class HfFolder mà thư viện evaluate đang tìm kiếm
class MockHfFolder:
    @classmethod
    def get_token(cls):
        return None  # Trả về None để evaluate bỏ qua bước check token

# 2. Tiêm (Inject) class này vào đúng "ngõ ngách" mà evaluate sẽ truy cập
huggingface_hub.hf_api.HfFolder = MockHfFolder

# 3. Bây giờ mới import và load evaluate
import evaluate
try:
    rouge = evaluate.load('rouge')
    print("Đã vá lỗi thành công! Bạn có thể chạy tiếp trainer.train().")
except Exception as e:
    print(f"Vẫn gặp vấn đề: {e}")

Đã vá lỗi thành công! Bạn có thể chạy tiếp trainer.train().


In [ ]:
def rouge_tokenize(text):
    import re
    import unicodedata
    return re.findall(r"[^\W_]+", unicodedata.normalize("NFC", str(text)).casefold())


def to_dataset(rows):
    keep = ['id', 'url', 'category', 'quality_tag', 'source', 'summary', 'src_word_count', 'tgt_word_count', 'length_ratio']
    clean = [{k: r.get(k, '') for k in keep} for r in rows if r.get('source') and r.get('summary')]
    return Dataset.from_list(clean)

ds = DatasetDict({
    'train': to_dataset(train_stage2),
    'val': to_dataset(val_raw),
    'test': to_dataset(test_raw),
})
print(ds)

tokenizer = AutoTokenizer.from_pretrained(CONFIG['STAGE1_CKPT'])
model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG['STAGE1_CKPT'])
print(f"Tokenizer: {type(tokenizer).__name__}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

def preprocess(batch):
    model_inputs = tokenizer(
        batch['source'],
        max_length=CONFIG['MAX_INPUT_LEN'],
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=batch['summary'],
        max_length=CONFIG['MAX_TARGET_LEN'],
        truncation=True,
        padding=False,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tok_ds = ds.map(preprocess, batched=True, remove_columns=ds['train'].column_names)
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding='longest',
    label_pad_token_id=-100,
)

rouge = evaluate.load('rouge')

def normalize_for_rouge(text):
    return '\n'.join(str(text).strip().split())

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    pred_text = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_text = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(tokenizer=rouge_tokenize, rouge_types=["rouge1", "rouge2", "rougeL"], 
        predictions=[normalize_for_rouge(x) for x in pred_text],
        references=[normalize_for_rouge(x) for x in label_text],
        use_stemmer=False,
    )
    return {k: round(v * 100, 4) for k, v in result.items()}

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=CONFIG['EPOCHS'],
    learning_rate=CONFIG['LR'],
    per_device_train_batch_size=CONFIG['TRAIN_BATCH'],
    per_device_eval_batch_size=CONFIG['EVAL_BATCH'],
    gradient_accumulation_steps=CONFIG['GRAD_ACCUM'],
    weight_decay=CONFIG['WEIGHT_DECAY'],
    warmup_ratio=CONFIG['WARMUP_RATIO'],
    label_smoothing_factor=CONFIG['LABEL_SMOOTH'],
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=CONFIG['GEN_MAX_LEN'],
    generation_num_beams=CONFIG['NUM_BEAMS'],
    bf16=USE_BF16,
    fp16=USE_FP16,
    logging_steps=25,
    report_to='none',
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_ds['train'],
    eval_dataset=tok_ds['val'],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG['PATIENCE'])],
)

train_result = trainer.train()
trainer.save_state()
trainer.save_model(str(BEST_DIR))
tokenizer.save_pretrained(str(BEST_DIR))

with open(RESULTS_DIR / 'train_result.json', 'w', encoding='utf-8') as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)
print('Best model saved:', BEST_DIR)

## Cell 7 - Evaluation SFT

In [ ]:
def rouge_tokenize(text):
    import re
    import unicodedata
    return re.findall(r"[^\W_]+", unicodedata.normalize("NFC", str(text)).casefold())


def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForSeq2SeqLM.from_pretrained(str(BEST_DIR)).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(str(BEST_DIR))
model.eval()
clear_cuda()

def number_recall(src, pred):
    src_nums = extract_numbers(src)
    if not src_nums:
        return 1.0
    return len(src_nums & extract_numbers(pred)) / len(src_nums)

def number_hallucination(src, pred):
    pred_nums = extract_numbers(pred)
    if not pred_nums:
        return 0.0
    return len(pred_nums - extract_numbers(src)) / len(pred_nums)

def length_ratio(src, pred):
    return len(str(pred).split()) / max(len(str(src).split()), 1)

def length_compliance(src, pred):
    ratio = length_ratio(src, pred)
    return float(CONFIG['MIN_RATIO'] <= ratio <= CONFIG['MAX_RATIO'])

def generate_summaries(texts, batch_size=None, **gen_kwargs):
    batch_size = batch_size or CONFIG['EVAL_BATCH']
    outputs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(
            batch_texts,
            max_length=CONFIG['MAX_INPUT_LEN'],
            truncation=True,
            padding=True,
            return_tensors='pt',
        ).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(
                **inputs,
                max_length=CONFIG['GEN_MAX_LEN'],
                min_length=CONFIG['GEN_MIN_LEN'],
                num_beams=CONFIG['NUM_BEAMS'],
                no_repeat_ngram_size=CONFIG['NO_REPEAT_NGRAM'],
                length_penalty=CONFIG['LENGTH_PENALTY'],
                early_stopping=True,
                **gen_kwargs,
            )
        outputs.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
    return outputs

from bert_score import score as bertscore

def evaluate_predictions(srcs, golds, preds, tag='SFT'):
    rouge_result = rouge.compute(tokenizer=rouge_tokenize, rouge_types=["rouge1", "rouge2", "rougeL"], 
        predictions=[normalize_for_rouge(x) for x in preds],
        references=[normalize_for_rouge(x) for x in golds],
        use_stemmer=False,
    )
    _, _, bert_f1 = bertscore(
        preds,
        golds,
        lang='vi',
        model_type='xlm-roberta-base',
        verbose=False,
    )
    nr = [number_recall(s, p) for s, p in zip(srcs, preds)]
    nh = [number_hallucination(s, p) for s, p in zip(srcs, preds)]
    lc = [length_compliance(s, p) for s, p in zip(srcs, preds)]
    lr = [length_ratio(s, p) for s, p in zip(srcs, preds)]
    return {
        'tag': tag,
        'n_samples': len(preds),
        'rouge1': round(rouge_result['rouge1'] * 100, 4),
        'rouge2': round(rouge_result['rouge2'] * 100, 4),
        'rougeL': round(rouge_result['rougeL'] * 100, 4),
        'bertscore_f1': round(float(bert_f1.mean().item()) * 100, 4),
        'number_recall': round(float(np.mean(nr)) * 100, 4),
        'number_hallucination': round(float(np.mean(nh)) * 100, 4),
        'length_compliance': round(float(np.mean(lc)) * 100, 4),
        'length_ratio_mean': round(float(np.mean(lr)), 4),
    }

test_src = list(ds['test']['source'])
test_gold = list(ds['test']['summary'])
test_ids = list(ds['test']['id'])

preds_sft = generate_summaries(test_src, batch_size=CONFIG['EVAL_BATCH'])
metrics_sft = evaluate_predictions(test_src, test_gold, preds_sft, tag='SFT')
print(pd.DataFrame([metrics_sft]).to_string(index=False))

sft_records = [
    {'id': test_ids[i], 'source': test_src[i], 'gold': test_gold[i], 'pred_sft': preds_sft[i]}
    for i in range(len(test_src))
]
with open(RESULTS_DIR / 'eval_sft.json', 'w', encoding='utf-8') as f:
    json.dump({'metrics': metrics_sft, 'records': sft_records}, f, ensure_ascii=False, indent=2)

## Cell 8 - Candidate Generation & Reranking

In [ ]:
def dedupe_keep_order(items):
    seen = set()
    out = []
    for item in items:
        key = ' '.join(str(item).split())
        if key and key not in seen:
            seen.add(key)
            out.append(item)
    return out

def generate_candidates(texts, k=None, batch_size=4):
    k = k or CONFIG['NUM_CANDIDATES']
    all_candidates = []
    num_beams = max(k * 2, k)
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(
            batch_texts,
            max_length=CONFIG['MAX_INPUT_LEN'],
            truncation=True,
            padding=True,
            return_tensors='pt',
        ).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(
                **inputs,
                max_length=CONFIG['GEN_MAX_LEN'],
                min_length=CONFIG['GEN_MIN_LEN'],
                num_beams=num_beams,
                num_beam_groups=k, # Chia các chùm (beams) thành k nhóm khác nhau
                num_return_sequences=k,
                diversity_penalty=CONFIG['DIVERSITY_PENALTY'], # Áp dụng hình phạt đa dạng
                no_repeat_ngram_size=CONFIG['NO_REPEAT_NGRAM'],
                length_penalty=CONFIG['LENGTH_PENALTY'],
                early_stopping=True,
            )
        decoded = tokenizer.batch_decode(ids, skip_special_tokens=True)
        for j in range(len(batch_texts)):
            cands = dedupe_keep_order(decoded[j * k:(j + 1) * k])
            while len(cands) < k:
                cands.append(cands[-1] if cands else '')
            all_candidates.append(cands[:k])
    return all_candidates

def length_score(src, pred):
    ratio = length_ratio(src, pred)
    if CONFIG['MIN_RATIO'] <= ratio <= CONFIG['MAX_RATIO']:
        return 1.0
    return max(0.0, 1.0 - abs(ratio - CONFIG['TARGET_RATIO']) / CONFIG['TARGET_RATIO'])

def score_candidate(src, cand):
    return (
        CONFIG['W_NUMBER'] * number_recall(src, cand)
        + CONFIG['W_LENGTH'] * length_score(src, cand)
        - CONFIG['W_HALLUCINATION'] * number_hallucination(src, cand)
    )

def rerank_and_select(srcs, candidate_lists):
    picked = []
    all_scores = []
    for src, cands in zip(srcs, candidate_lists):
        scores = [float(score_candidate(src, cand)) for cand in cands]
        best_idx = int(np.argmax(scores))
        picked.append(cands[best_idx])
        all_scores.append(scores)
    return picked, all_scores

candidates = generate_candidates(test_src, k=CONFIG['NUM_CANDIDATES'], batch_size=4)
preds_rerank, candidate_scores = rerank_and_select(test_src, candidates)

print('Sample candidates for first test item:')
for cand, score in zip(candidates[0], candidate_scores[0]):
    print(f'[{score:.4f}] {cand[:220]}')

## Cell 9 - Final Evaluation & Export

In [12]:
metrics_rerank = evaluate_predictions(test_src, test_gold, preds_rerank, tag='SFT+Rerank')
metrics_df = pd.DataFrame([metrics_sft, metrics_rerank])
display(metrics_df)

records = []
for i in range(len(test_src)):
    records.append({
        'id': test_ids[i],
        'source': test_src[i],
        'gold': test_gold[i],
        'pred_sft': preds_sft[i],
        'pred_rerank': preds_rerank[i],
        'candidates': candidates[i],
        'candidate_scores': candidate_scores[i],
        'sft_number_recall': number_recall(test_src[i], preds_sft[i]),
        'rerank_number_recall': number_recall(test_src[i], preds_rerank[i]),
        'sft_length_ratio': length_ratio(test_src[i], preds_sft[i]),
        'rerank_length_ratio': length_ratio(test_src[i], preds_rerank[i]),
    })

final_results = {
    'model': CONFIG['MODEL_LABEL'],
    'stage1_ckpt': CONFIG['STAGE1_CKPT'],
    'best_stage2_dir': str(BEST_DIR),
    'data_dir': str(DATA_DIR),
    'train_size': len(ds['train']),
    'val_size': len(ds['val']),
    'test_size': len(ds['test']),
    'config': CONFIG,
    'metrics': {
        'sft': metrics_sft,
        'sft_rerank': metrics_rerank,
    },
    'records': records,
}

with open(RESULTS_DIR / 'final_results_stage2.json', 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

with open(RESULTS_DIR / 'stage2_predictions.jsonl', 'w', encoding='utf-8') as f:
    for row in records:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

metrics_df.to_csv(RESULTS_DIR / 'stage2_metrics.csv', index=False)
print('Saved:')
print(' -', RESULTS_DIR / 'final_results_stage2.json')
print(' -', RESULTS_DIR / 'stage2_predictions.jsonl')
print(' -', RESULTS_DIR / 'stage2_metrics.csv')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,tag,n_samples,rouge1,rouge2,rougeL,bertscore_f1,number_recall,number_hallucination,length_compliance,length_ratio_mean
0,SFT,105,81.0698,69.7006,68.6854,92.4743,71.9750,6.5946,32.381,0.2030
1,SFT+Rerank,105,83.1606,71.6800,68.9095,92.5069,78.4657,5.1894,47.619,0.2123


Saved:
 - /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_2/runs/vit5_base_stage2/results/final_results_stage2.json
 - /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_2/runs/vit5_base_stage2/results/stage2_predictions.jsonl
 - /content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP/ViT5_stage_2/runs/vit5_base_stage2/results/stage2_metrics.csv


In [13]:
import random

def run_demo_and_export(n_samples=5, filename='inference_demo_results.txt'):
    # 1. Xác định đường dẫn lưu file trên Drive
    # Biến RESULTS_DIR đã được định nghĩa ở các cell cấu hình trước đó
    output_path = RESULTS_DIR / filename

    # 2. Lấy mẫu ngẫu nhiên từ tập test
    indices = random.sample(range(len(ds['test'])), n_samples)

    header = f"--- DEMO CHAT LUONG TOM TAT (TAP TEST) ---\n"
    header += f"Model: {CONFIG['MODEL_LABEL']}\n"
    header += "-" * 50 + "\n"

    print(header)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(header)

        for idx in indices:
            item = ds['test'][idx]
            source_text = item['source']
            reference_summary = item['summary']

            # 3. Chuẩn bị đầu vào cho Model
            inputs = tokenizer(
                source_text,
                return_tensors="pt",
                max_length=CONFIG['MAX_INPUT_LEN'],
                truncation=True
            ).to(DEVICE)

            # 4. Sinh tóm tắt
            with torch.no_grad():
                summary_ids = model.generate(
                    inputs["input_ids"],
                    max_length=CONFIG['GEN_MAX_LEN'],
                    min_length=CONFIG['GEN_MIN_LEN'],
                    num_beams=CONFIG['NUM_BEAMS'],
                    no_repeat_ngram_size=CONFIG['NO_REPEAT_NGRAM'],
                    length_penalty=CONFIG['LENGTH_PENALTY'],
                    early_stopping=True
                )

            model_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

            # 5. Định dạng nội dung mẫu
            ref_len = len(reference_summary.split())
            mod_len = len(model_summary.split())

            sample_output = f"\n[Mau so {idx}]\n"
            sample_output += f"URL: {item.get('url', 'N/A')}\n"
            sample_output += f"\nO: NOI DUNG GOC (Trích 300 ky tu dau):\n"
            sample_output += f"   {source_text[:300]}...\n"
            sample_output += f"\nR: TOM TAT GOC (Ground Truth):\n"
            sample_output += f"   {reference_summary}\n"
            sample_output += f"\nM: AI TOM TAT (Model Output):\n"
            sample_output += f"   {model_summary}\n"
            sample_output += f"\n>> Do dai: Goc ({ref_len} tu) | AI ({mod_len} tu)\n"
            sample_output += "-" * 80 + "\n"

            # In ra console
            print(sample_output)
            # Ghi vào file
            f.write(sample_output)

    print(f"✅ Da xuat ket qua thanh cong vao Drive: {output_path}")

# Chạy demo và xuất file
run_demo_and_export(n_samples=5)

--- DEMO CHAT LUONG TOM TAT (TAP TEST) ---
Model: ViT5-base self Stage1 -> Stage2
--------------------------------------------------


[Mau so 81]
URL: https://vnexpress.net/ap-xe-nguc-kem-ung-thu-vu-5059852.html

O: NOI DUNG GOC (Trích 300 ky tu dau):
   TP HCM Chị Khanh, 45 tuổi, đau nhức ngực, bác sĩ chẩn đoán áp xe ngực trái, ung thư vú bên phải, được phẫu thuật triệt căn ung thư kết hợp tái tạo ngực.
Kết quả siêu âm của chị Khanh tại Phòng khám Đa khoa Tâm Anh Quận 7 cho thấy ngực trái viêm áp xe, ngực phải có tổn thương hình dạng không đều kích...

R: TOM TAT GOC (Ground Truth):
   TP HCM Chị Khanh, 45 tuổi, đau nhức ngực, bác sĩ chẩn đoán áp xe ngực trái, ung thư vú bên phải, được phẫu thuật triệt căn ung thư kết hợp tái tạo ngực.
Kết quả siêu âm của chị Khanh tại Phòng khám Đa khoa Tâm Anh Quận 7 cho thấy ngực trái viêm áp xe, ngực phải có tổn thương hình dạng không đều kích thước 9x12 mm, siêu âm đàn hồi cứng, đánh giá BIRADS-4B (nghi ngờ ung thư).
Khi vết thương lành lặn, ngư